In [1]:
import re
import pandas as pd
import io

In [2]:
with open("support_logs_2025-07-01.log",encoding = 'utf-8') as f:
       content = f.read()
len(content)

32938

In [3]:
lines = content.split("---")
lines

['2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000\nIP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false\nUserAgent="PostmanRuntime/7.32.2"\nMessage=" event for TCK0701000"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None\n',
 '\n2025-07-01 00:41:00 [INFO] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000\nIP=58.36.189.27 | ResponseTime=1505ms | CPU=57.24% | EventType=generic_event | Error=false\nUserAgent="Mobile-Safari/537.36"\nMessage=" event for TCK0701000"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None\n',
 '\n2025-07-01 01:44:00 [DEBUG] careplus.support.GenericService - TicketID=TCK0701001 SessionID=sess_TCK0701001\nIP=181.18.12.170 | ResponseTime=586ms | CPU=78.43% | EventType=generic_event | Error=false\nUserAgent="curl/7.68.0"\nMessage=" event for TCK0701001"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None\n',
 '\n2025-07-01 01:49:00 [DEBUG] careplus.su

In [4]:
entries = [entry.strip() for entry in content.split("---")]
print(entries[0])

2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000
IP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false
UserAgent="PostmanRuntime/7.32.2"
Message=" event for TCK0701000"
Debug="ℹ️ Logged for monitoring"
TraceID=None


In [5]:
import re

log = '''2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000 IP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false UserAgent="PostmanRuntime/7.32.2" Message=" event for TCK0701000" Debug="ℹ️ Logged for monitoring" TraceID=None'''

pattern = r'''
(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})
\s+\[(?P<level>[^\]]+)\]
\s+(?P<service>\S+)
\s+-\s+
TicketID=(?P<ticket_id>\S+)
\s+SessionID=(?P<session_id>\S+)
\s+IP=(?P<ip>[\d.]+)
\s*\|\s*
ResponseTime=(?P<response_time>\d+)ms
\s*\|\s*
CPU=(?P<cpu>[\d.]+)%
\s*\|\s*
EventType=(?P<event_type>\S+)
\s*\|\s*
Error=(?P<error>\w+)
\s+UserAgent="(?P<user_agent>[^"]*)"
\s+Message="(?P<message>[^"]*)"
\s+Debug="(?P<debug>[^"]*)"
\s+TraceID=(?P<trace_id>\S+)
'''

match = re.search(pattern, log, re.VERBOSE)
lst = []

if match:
    lst.append(match.groupdict())

    for key, value in lst[0].items():
        print(f"{key}: {value}")

else:
    print("No match found!")

timestamp: 2025-07-01 00:21:00
level: INF0
service: careplus.support.GenericService
ticket_id: TCK0701000
session_id: sess_TCK0701000
ip: 60.130.155.7
response_time: 1269
cpu: 27.64
event_type: generic_event
error: false
user_agent: PostmanRuntime/7.32.2
message:  event for TCK0701000
debug: ℹ️ Logged for monitoring
trace_id: None


In [6]:
parsed_entries = []

entries = log.split("---")

for entry in entries:
    entry = entry.strip()

    match = re.search(pattern, entry, re.VERBOSE)

    if match:
        parsed_entries.append(match.groupdict())

parsed_entries

[{'timestamp': '2025-07-01 00:21:00',
  'level': 'INF0',
  'service': 'careplus.support.GenericService',
  'ticket_id': 'TCK0701000',
  'session_id': 'sess_TCK0701000',
  'ip': '60.130.155.7',
  'response_time': '1269',
  'cpu': '27.64',
  'event_type': 'generic_event',
  'error': 'false',
  'user_agent': 'PostmanRuntime/7.32.2',
  'message': ' event for TCK0701000',
  'debug': 'ℹ️ Logged for monitoring',
  'trace_id': 'None'}]

In [7]:
df = pd.DataFrame(parsed_entries)

In [8]:
df.head(5)

,timestamp,level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring,None


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   timestamp      1 non-null      str  
 1   level          1 non-null      str  
 2   service        1 non-null      str  
 3   ticket_id      1 non-null      str  
 4   session_id     1 non-null      str  
 5   ip             1 non-null      str  
 6   response_time  1 non-null      str  
 7   cpu            1 non-null      str  
 8   event_type     1 non-null      str  
 9   error          1 non-null      str  
 10  user_agent     1 non-null      str  
 11  message        1 non-null      str  
 12  debug          1 non-null      str  
 13  trace_id       1 non-null      str  
dtypes: str(14)
memory usage: 244.0 bytes


In [10]:
df.drop("trace_id",axis = 1)
df.head(2)

,timestamp,level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring,None


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   timestamp      1 non-null      str  
 1   level          1 non-null      str  
 2   service        1 non-null      str  
 3   ticket_id      1 non-null      str  
 4   session_id     1 non-null      str  
 5   ip             1 non-null      str  
 6   response_time  1 non-null      str  
 7   cpu            1 non-null      str  
 8   event_type     1 non-null      str  
 9   error          1 non-null      str  
 10  user_agent     1 non-null      str  
 11  message        1 non-null      str  
 12  debug          1 non-null      str  
 13  trace_id       1 non-null      str  
dtypes: str(14)
memory usage: 244.0 bytes


In [12]:
df = df.astype({

    "cpu" : 'float',
    "response_time" : 'int',
    
})


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      1 non-null      str    
 1   level          1 non-null      str    
 2   service        1 non-null      str    
 3   ticket_id      1 non-null      str    
 4   session_id     1 non-null      str    
 5   ip             1 non-null      str    
 6   response_time  1 non-null      int64  
 7   cpu            1 non-null      float64
 8   event_type     1 non-null      str    
 9   error          1 non-null      str    
 10  user_agent     1 non-null      str    
 11  message        1 non-null      str    
 12  debug          1 non-null      str    
 13  trace_id       1 non-null      str    
dtypes: float64(1), int64(1), str(12)
memory usage: 244.0 bytes


In [14]:
df.describe()

,response_time,cpu
count,1.0,1.00
mean,1269.0,27.64
std,NaN,NaN
min,1269.0,27.64
25%,1269.0,27.64
50%,1269.0,27.64
75%,1269.0,27.64
max,1269.0,27.64


In [15]:
df["level"].value_counts()

level
INF0    1
Name: count, dtype: int64

In [16]:
import re
import pandas as pd

with open("support_logs_2025-07-01.log", "r", encoding="utf-8") as file:
    log_data = file.read()

# Extract every log level
levels = re.findall(r'\[([^\]]+)\]', log_data)

# Convert to DataFrame
level_df = pd.DataFrame({"level": levels})

print(level_df["level"].value_counts())

level
INFO       40
DEBUG      36
INF0       13
DEBG       10
warnING     3
WARNING     3
Name: count, dtype: int64


In [17]:
level_mapping = {
    "INF0": "INFO",
    "DEBG": "DEBUG",
    "warnING": "WARNING"
}

df["level"] = df["level"].replace(level_mapping)

print(df["level"].value_counts())

level
INFO    1
Name: count, dtype: int64


In [18]:
df[df.duplicated()]

,timestamp,level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id


In [19]:
df = df.drop_duplicates()
df[df.duplicated()]

,timestamp,level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id


In [20]:
df.shape

(1, 14)

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      1 non-null      str    
 1   level          1 non-null      str    
 2   service        1 non-null      str    
 3   ticket_id      1 non-null      str    
 4   session_id     1 non-null      str    
 5   ip             1 non-null      str    
 6   response_time  1 non-null      int64  
 7   cpu            1 non-null      float64
 8   event_type     1 non-null      str    
 9   error          1 non-null      str    
 10  user_agent     1 non-null      str    
 11  message        1 non-null      str    
 12  debug          1 non-null      str    
 13  trace_id       1 non-null      str    
dtypes: float64(1), int64(1), str(12)
memory usage: 244.0 bytes
